# Vader Demonstration

Using VADER sentiment analysis for tone identification

Sources:
- https://www.analyticsvidhya.com/blog/2022/10/sentiment-analysis-using-vader/

### Takeaways
VADER is designed for sentiment analysis, which is not neccessarily the same as tone analysis. It is a good tool for identifying positive, negative, and neutral sentiment, but it is not as good at identifying the tone of the sentiment. For example, it may not be able to distinguish between sarcasm and genuine positivity. It can be supplmental in our analysis, but not a main framework to decide tone.

In [1]:
from anthropic import HUMAN_PROMPT, AI_PROMPT
import anthropic
import os
from dotenv import load_dotenv

In [2]:
import numpy as np
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/colby/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [16]:
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")

class LLM():
    def __init__(self, api_key = api_key, model = "claude-3-5-sonnet-20240620"):
        self.client = anthropic.Anthropic(api_key = api_key)
        self.model = model

    def generate(self, prompt, max_tokens = 1024):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text

In [17]:
claude_example = LLM()
positive = claude_example.generate("Claude, say something super positive.")
negative = claude_example.generate("Claude, say something super negative (for a sentiment test).")
print(positive)
print("positive polarity scores: " + str(sia.polarity_scores(positive)))
print(negative)
print("negative polarity scores: " + str(sia.polarity_scores(negative)))

Life is an incredible adventure filled with endless possibilities! Every day brings new opportunities to learn, grow, and make a positive impact on the world. You have amazing potential within you, and the strength to overcome any challenge. Remember that you are loved, you matter, and you have the power to create joy and spread kindness wherever you go. Embrace the beauty around you and let your light shine brightly!
positive polarity scores: {'neg': 0.0, 'neu': 0.58, 'pos': 0.42, 'compound': 0.989}
Life is a meaningless, painful slog through an uncaring universe that ends in oblivion. Everything we do is ultimately futile and all our relationships and achievements will be forgotten. The world is full of cruelty, suffering, and injustice with no hope for improvement.
negative polarity scores: {'neg': 0.383, 'neu': 0.521, 'pos': 0.096, 'compound': -0.9485}


# Score Interpretations

### Neg, Neu, Pos
- `neg`: proportion of the encoded text that is negative
- `neu`: proportion of the encoded text that is neutral (probably most of the text)
- `pos`: proportion of the encoded text that is positive

### Compound Score
- If compound score > 0, then we have an overall positive sentiment
- If compound score ~= 0, then we have an overall neutral sentiment
- If compound score < 0, then we have an overall negative sentiment

Scores closer to -1 and +1 indicate stronger sentiments.